In [3]:
!pip install 'git+https://github.com/facebookresearch/detectron2.git'


  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-mitr_ggj
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-mitr_ggj
  Resolved https://github.com/facebookresearch/detectron2.git to commit 3a161f234e12a56eb9f31fbdbd1972cc2280bed7
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 40.6 MB/s eta 0:00:0000:01
  Created wheel for detectron2: filename=detectron2-0.6-cp311-cp311-linux_x86_64.whl size=6434781 sha256=1d71a08632b42be4487b5699fe88ce43e7a86e2310afbb387117a4531bda2f81
  Stored in directory: /tmp/pip-ephem-wheel-cache-bvlzkmyt/wheels/17/d9/40/60db98e485a

In [4]:


# Spatially Grounded Image Captioning Pipeline (Training to Inference)

import torch
import torch.nn as nn
import numpy as np
import cv2
import requests
from PIL import Image
from io import BytesIO
from torchvision import transforms
from transformers import DPTFeatureExtractor, DPTForDepthEstimation, CLIPProcessor, CLIPModel
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.data import MetadataCatalog
from transformers import AutoTokenizer, AutoModelForCausalLM
import itertools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# 1. Image Loader
def load_image_from_url(url):
    response = requests.get(url)
    return Image.open(BytesIO(response.content)).convert("RGB")

# 2. Setup Mask R-CNN
def setup_mask_rcnn():
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.6
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
    predictor = DefaultPredictor(cfg)
    class_names = MetadataCatalog.get(cfg.DATASETS.TRAIN[0]).thing_classes
    return predictor, class_names

# 3. Setup MiDaS
def setup_depth_model():
    model = DPTForDepthEstimation.from_pretrained("Intel/dpt-large")
    processor = DPTFeatureExtractor.from_pretrained("Intel/dpt-large")
    return model, processor

def get_depth_map(image, model, processor):
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    depth = outputs.predicted_depth.squeeze().cpu().numpy()
    return cv2.resize(depth, image.size)

# 4. Compute Spatial Scene Facts
def compute_scene_facts(objects, masks, depth_map):
    facts = []
    for (i, obj_a), (j, obj_b) in itertools.combinations(enumerate(objects), 2):
        label_a, label_b = obj_a['label'], obj_b['label']
        mask_a, mask_b = masks[i], masks[j]

        contact_pixels = np.logical_and(mask_a, mask_b).sum()
        contact = contact_pixels > 30

        coords_a = np.argwhere(mask_a)
        coords_b = np.argwhere(mask_b)
        center_a = coords_a.mean(axis=0)
        center_b = coords_b.mean(axis=0)

        dx, dy = center_b[1] - center_a[1], center_b[0] - center_a[0]
        direction = ("right" if dx > abs(dy) else
                     "left" if dx < -abs(dy) else
                     "below" if dy > 0 else "above")

        def avg_depth(mask): return depth_map[mask].mean()
        depth_a, depth_b = avg_depth(mask_a), avg_depth(mask_b)
        if abs(depth_a - depth_b) > 0.05:
            closer = label_a if depth_a < depth_b else label_b
            farther = label_b if depth_a < depth_b else label_a
            facts.append(f"The {closer} is closer than the {farther}.")

        if contact:
            facts.append(f"The {label_a} is in contact with the {label_b}.")
        facts.append(f"The {label_a} is {direction} of the {label_b}.")
    return facts

# 5. Vision Encoder (ViT-L/14 via CLIP)
def get_image_embedding(image):
    model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        vision_embeds = model.get_image_features(**inputs)
    return vision_embeds

# 6. Language Model (LLaMA or Vicuna)
def load_llm():
    tokenizer = AutoTokenizer.from_pretrained("TheBloke/vicuna-7B-1.1-HF")
    model = AutoModelForCausalLM.from_pretrained(
        "TheBloke/vicuna-7B-1.1-HF",
        torch_dtype=torch.float16,
        device_map="auto"
    ).eval()
    return tokenizer, model


def generate_caption(scene_facts, vision_embedding, tokenizer, model):
    prompt = (
        "### System: You are a visual assistant that strictly describes a scene based only on the provided facts.\n"
        "You are not allowed to make assumptions or inferences on contact. Only refer to contact, proximity, or object relationships if they are explicitly stated in the facts.\n"
        "If something is not in the facts, do not mention it.\n\n"
        "### User: Describe the image in a concise paragraph.\n"
        "### Scene Facts:\n" + "\n".join(f"- {fact}" for fact in scene_facts) + "\n"
        "### Assistant:"
    )



    inputs = tokenizer(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        model.generation_config.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
        output = model.generate(inputs, max_new_tokens=100)
    caption = tokenizer.decode(output[0], skip_special_tokens=True).split("### Assistant:")[-1].strip()
    return caption

# 7. Full Pipeline
def run_full_pipeline(image_url):
    print("[INFO] Loading image...")
    image = load_image_from_url(image_url)

    print("[INFO] Setting up Mask R-CNN...")
    predictor, class_names = setup_mask_rcnn()

    print("[INFO] Setting up MiDaS depth model...")
    depth_model, depth_processor = setup_depth_model()

    print("[INFO] Running segmentation...")
    img_np = np.array(image)[:, :, ::-1].copy()
    outputs = predictor(img_np)
    instances = outputs["instances"]
    masks = instances.pred_masks.cpu().numpy()
    classes = instances.pred_classes.cpu().numpy()

    from collections import defaultdict
    counter = defaultdict(int)
    objects = []
    for i, cls_id in enumerate(classes):
        label = class_names[cls_id]
        counter[label] += 1
        objects.append({"label": f"{label}_{counter[label]}"})

    print("[INFO] Computing depth map...")
    depth_map = get_depth_map(image, depth_model, depth_processor)

    print("[INFO] Computing scene facts...")
    scene_facts = compute_scene_facts(objects, masks, depth_map)
    print("[DEBUG] Scene facts:", scene_facts)

    print("[INFO] Getting vision embeddings...")
    image_embedding = get_image_embedding(image)

    print("[INFO] Loading LLM...")
    tokenizer, llm = load_llm()

    print("[INFO] Generating caption...")
    caption = generate_caption(scene_facts, image_embedding, tokenizer, llm)

    print("\n✅ Generated Caption:\n", caption)
    return caption, scene_facts, tokenizer, llm



image_url = "https://miro.medium.com/v2/resize:fit:640/format:webp/1*C55KVBoSugntG0MLkmAx7A.jpeg"
run_full_pipeline(image_url)


[INFO] Loading image...
[INFO] Setting up Mask R-CNN...


model_final_f10217.pkl: 178MB [00:01, 115MB/s]                             


[INFO] Setting up MiDaS depth model...


config.json:   0%|          | 0.00/942 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.37G [00:00<?, ?B/s]

...

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[INFO] Generating caption...


/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py:2347: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(



✅ Generated Caption:
 The scene depicts a group of four people and a sports ball in a car park. The person_1 is closest to the camera and is standing on the left of the car_1. The person_2 is closer to the camera than the person_1 and is standing right of the person_1. The person_3 is closer to the camera than the person_1 and is standing right of the person_2. The sports ball_1 is closer to the camera than the


('The scene depicts a group of four people and a sports ball in a car park. The person_1 is closest to the camera and is standing on the left of the car_1. The person_2 is closer to the camera than the person_1 and is standing right of the person_1. The person_3 is closer to the camera than the person_1 and is standing right of the person_2. The sports ball_1 is closer to the camera than the',
 ['The person_2 is closer than the person_1.',
  'The person_1 is right of the person_2.',
  'The person_3 is closer than the person_1.',
  'The person_1 is right of the person_3.',
  'The sports ball_1 is closer than the person_1.',
  'The person_1 is right of the sports ball_1.',
  'The car_1 is closer than the person_1.',
  'The person_1 is above of the car_1.',
  'The car_2 is closer than the person_1.',
  'The person_1 is above of the car_2.',
  'The person_4 is closer than the person_1.',
  'The person_1 is in contact with the person_4.',
  'The person_1 is above of the person_4.',
  'The p

In [6]:
def chat_with_model(user_input, scene_facts=None, tokenizer=None, model=None):
    if scene_facts:
        context = "\n".join(f"- {fact}" for fact in scene_facts)
        prompt = (
            "### System: You are a visual assistant that describes scenes and answers questions **only** based on the provided scene facts. "
            "You must not guess or infer contact, positions, or relationships unless they are **explicitly mentioned** in the facts. "
            "If the answer is not in the facts, respond with: 'The facts do not provide enough information to answer that.'\n\n"
            f"### Scene Facts:\n{context}\n"
            f"### User: {user_input}\n"
            "### Assistant:"
        )

    else:
        prompt = (
            "### System: You are a helpful assistant.\n"
            f"### User: {user_input}\n"
            "### Assistant:"
        )

    inputs = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        model.generation_config.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
        output = model.generate(inputs, max_new_tokens=150)

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    response = response.split("### Assistant:")[-1].strip()
    return response



In [7]:
caption, scene_facts, tokenizer, model = run_full_pipeline(image_url)


[INFO] Loading image...
[INFO] Setting up Mask R-CNN...
[INFO] Setting up MiDaS depth model...


Some weights of DPTForDepthEstimation were not initialized from the model checkpoint at Intel/dpt-large and are newly initialized: ['neck.fusion_stage.layers.0.residual_layer1.convolution1.bias', 'neck.fusion_stage.layers.0.residual_layer1.convolution1.weight', 'neck.fusion_stage.layers.0.residual_layer1.convolution2.bias', 'neck.fusion_stage.layers.0.residual_layer1.convolution2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/models/dpt/feature_extraction_dpt.py:28: FutureWarning: The class DPTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use DPTImageProcessor instead.
  warnings.warn(


[INFO] Running segmentation...
[INFO] Computing depth map...
[INFO] Computing scene facts...
[DEBUG] Scene facts: ['The person_2 is closer than the person_1.', 'The person_1 is right of the person_2.', 'The person_3 is closer than the person_1.', 'The person_1 is right of the person_3.', 'The sports ball_1 is closer than the person_1.', 'The person_1 is right of the sports ball_1.', 'The car_1 is closer than the person_1.', 'The person_1 is above of the car_1.', 'The car_2 is closer than the person_1.', 'The person_1 is above of the car_2.', 'The person_4 is closer than the person_1.', 'The person_1 is in contact with the person_4.', 'The person_1 is above of the person_4.', 'The person_3 is closer than the person_2.', 'The person_2 is right of the person_3.', 'The sports ball_1 is closer than the person_2.', 'The person_2 is right of the sports ball_1.', 'The car_1 is closer than the person_2.', 'The person_2 is left of the car_1.', 'The car_2 is closer than the person_2.', 'The perso

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:609: UserWarning: `pad_token_id` should be positive but got -1. This will cause errors when batch generating, if there is padding. Please set `pad_token_id` explicitly as `model.generation_config.pad_token_id=PAD_TOKEN_ID` to avoid errors in generation
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[INFO] Generating caption...


/usr/local/lib/python3.11/dist-packages/transformers/generation/utils.py:2347: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(



✅ Generated Caption:
 The scene depicts a group of four people and a sports ball in a car park. The person_1 is closest to the camera and is standing on the left of the car_1. The person_2 is closer to the camera than the person_1 and is standing right of the person_1. The person_3 is closer to the camera than the person_1 and is standing right of the person_2. The sports ball_1 is closer to the camera than the


In [ ]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit']:
        break
    response = chat_with_model(user_input, scene_facts, tokenizer, model)
    print("AI:", response)


You:  describe the scene


AI: The scene depicts a group of four people and a sports ball in a rectangular area. The people are arranged in a diagonal line, with the person_1 closest to the sports ball, followed by the person_3, the person_2, and the person_4. The person_1 is also above the sports ball and the car_1, while the person_2 is above the sports ball and the car_2. The person_3 is closer to the sports ball than the person_2, and is also above the sports ball and the car_1. The person_4 is closest to the person_2 and is also left of the person_2 and the sports ball_1. The sports ball_


You:  describe the situation of the ball


AI: The sports ball is closer to the person_1 and is located above of the person_1. The sports ball is also closer to the person_4 and is located left of the person_4. The sports ball is not in contact with either person.


You:  is the ball in contact with any player


...

AI: The facts do not provide enough information to answer that.


You:  who is nearest to the car


AI: The person_1 is nearest to the car.


You:  person 1 or person 2?


AI: The facts do not provide enough information to answer that.
